# Notebook 03 — Machine shift on the final dataset: within-machine vs cross-machine (GPU, resume-safe)
**Run notebook 02 first** (it creates `prepared_v3`).
The study design (*Domain Shift and Domain Adaptation in Whole-Body Bone Scintigraphy*), Experiments 1 and 2:

| Exp | Train | Test |
|---|---|---|
| **1 — Within-machine** | camera A (5-fold CV inside camera A) | held-out fold of camera A |
| **2 — Cross-machine** | the *same* fold models | every patient of camera B |

Then the same with A and B swapped. Because Experiment 1 and 2 use **the same trained models**, the training-set size is
identical and the only difference is the camera of the test patients. A large drop from within to cross = machine (domain) shift.

**Design (fixed before running):**
* Patients: the audited classification patients of the final dataset (`prepared_v3`, notebook 02; Ind not used). Camera 1 (M1)
  and camera 2 (M2) are kept apart.
* Within camera A: 5 patient-level folds inside camera A (stratified by class, made once in notebook 02); inside every training set a
  class-stratified 15 % inner validation split for early stopping (same protocol, augmentation and hyper-parameters as the earlier revision run).
* Cross: each fold model predicts **all** patients of camera B (none of them was ever used for training).
* 5 tasks × 4 backbones × {ImageNet, BS-80K} × 2 cameras × 5 folds × 3 seeds = 1,200 runs (≈0.6 min each on a T4 → ≈12 h).
  Seed 42 for everything first (400 runs ≈ 4 h); seeds 43 and 44 afterwards.

**Analysis (last cells, CPU, works on partial results):** within vs cross macro-F1, balanced accuracy, AUC, sensitivity,
specificity, per-class recall and confusion matrices; drop = cross − within with a joint patient bootstrap (10,000 resamples),
Holm correction; pooled drop per initialisation; and whether the drop differs between ImageNet and BS-80K initialisation.

**Runtime:** T4 GPU → Run all. First `QUICK_TEST = True` (≈5 min), then `False`. Safe to re-run: finished runs are skipped.

In [ ]:
# settings
import os, sys, subprocess, time, copy, random, json, gc
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'huggingface_hub', 'python-docx'], check=False)
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
import timm
from PIL import Image
from sklearn.model_selection import train_test_split

LOCAL_TEST = bool(os.environ.get('BONESCAN_BASE'))
if LOCAL_TEST:
    BASE = os.environ['BONESCAN_BASE']
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Colab Notebooks/bonescan'
PREP, CV_IN, BSD = BASE + '/prepared', BASE + '/cv5_results', BASE + '/bs80k'
TIER2 = None                                     # combined-machine reference not available for the new dataset
OUT = BASE + '/revision/machine_v3'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'NOT FOUND - select Runtime > Change runtime type > GPU')

QUICK_TEST = True            # first True (short check), then False
MAX_HOURS = 11.5             # stop after this many hours; run all again to resume (finished runs are skipped)
RUN_TRAINING = True          # set False to re-run only the analysis

MODELS = ['tf_efficientnetv2_s', 'resnext50_32x4d', 'regnety_016', 'inception_v3']
TASKS = {
    '3class':              {'SECONDARY': 0, 'DEGENERATIVE': 1, 'NORMAL': 2},
    'cancer_vs_noncancer': {'SECONDARY': 1, 'DEGENERATIVE': 0, 'NORMAL': 0},
    'sec_vs_normal':       {'SECONDARY': 1, 'NORMAL': 0},
    'sec_vs_degen':        {'SECONDARY': 1, 'DEGENERATIVE': 0},
    'degen_vs_normal':     {'DEGENERATIVE': 1, 'NORMAL': 0},
}
TASK_ID = {'3class': 'T1', 'cancer_vs_noncancer': 'T2', 'sec_vs_normal': 'T3', 'sec_vs_degen': 'T4', 'degen_vs_normal': 'T5'}
MN = {'tf_efficientnetv2_s': 'EfficientNetV2-S', 'resnext50_32x4d': 'ResNeXt50', 'regnety_016': 'RegNetY-1.6GF', 'inception_v3': 'Inception-v3'}
SEEDS = [42, 43, 44]
INITS = ['IN', 'BS']                      # IN = ImageNet, BS = BS-80K
CAMERAS = ['M1', 'M2']
N_BOOT = 10000

N_FOLDS, EPOCHS, PATIENCE, BATCH, H, W = 5, 30, 8, 16, 512, 176
INNER_FRAC = 0.15
RAD_REPO, RAD_FILE = 'Lab-Rasool/RadImageNet', 'InceptionV3.pt'     # not used here (shared code)
PRETRAINED = not LOCAL_TEST

if QUICK_TEST:
    MODELS, TASKS = ['regnety_016'], {'sec_vs_normal': TASKS['sec_vs_normal'], '3class': TASKS['3class']}
    SEEDS, EPOCHS, N_BOOT = [42], 1, 200
    OUT = BASE + '/revision/machine_v3_quick'
QUICK_FOLDS = list(range(N_FOLDS))
for d in ['preds', 'logs', 'analysis']:
    os.makedirs(f'{OUT}/{d}', exist_ok=True)
RUNS = OUT + '/runs.csv'
T_START = time.time()
print('OUT:', OUT, '| QUICK_TEST:', QUICK_TEST)

In [ ]:
# data (prepared_v3 = the final dataset, made by notebook 02)
P3 = BASE + '/prepared_v3'
fix = lambda p: P3 + '/images/' + os.path.basename(str(p))
allp = pd.read_csv(P3 + '/patients.csv')
sp = allp[allp.split == 'pool'][['patient_id', 'machine', 'label', 'ANT', 'POST', 'fold']].reset_index(drop=True)
sp['fold'] = sp.fold.astype(int); sp['split'] = 'pool'      # no separate test set: 5-fold CV inside each machine
assert sp.patient_id.is_unique and sp.label.isin(['SECONDARY', 'DEGENERATIVE', 'NORMAL']).all()
print('classification patients:', len(sp), '| Ind not used:', (allp.split == 'ind').sum(), '| excluded by audit:', (allp.split == 'excluded').sum())
print(pd.crosstab(sp.label, sp.machine, margins=True))

ALL = sp[['patient_id', 'ANT', 'POST', 'machine']].reset_index(drop=True)
IDX = {p: i for i, p in enumerate(ALL.patient_id)}
CAM = dict(zip(ALL.patient_id, ALL.machine))
def load_img(p):
    return np.array(Image.open(fix(p)).convert('L'))
X = torch.from_numpy(np.stack([np.stack([load_img(a), load_img(b)]) for a, b in zip(ALL.ANT, ALL.POST)]))   # N x 2 x H x W uint8
assert X.shape[2:] == (H, W), X.shape
X = X.to(DEVICE)
print('images on', DEVICE, tuple(X.shape), f'{X.numel() / 1e6:.0f} MB')

In [ ]:
# model, GPU augmentation, training
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def augment(img, rng):
    """Same augmentation as the paper (RandomAffine ±7°, ±4 %, 0.93–1.07, white fill; brightness/contrast ±20 %), done on the GPU per image."""
    out = []
    for x in img:                                                   # x: 1 x H x W in [0,1]
        ang = float(rng.uniform(-7, 7)); sc = float(rng.uniform(0.93, 1.07))
        tx, ty = int(round(rng.uniform(-0.04, 0.04) * W)), int(round(rng.uniform(-0.04, 0.04) * H))
        x = TF.affine(x, angle=ang, translate=[tx, ty], scale=sc, shear=[0.0], interpolation=InterpolationMode.NEAREST, fill=[1.0])
        ops = [('b', float(rng.uniform(0.8, 1.2))), ('c', float(rng.uniform(0.8, 1.2)))]
        if rng.random() < 0.5:
            ops = ops[::-1]
        for k, f in ops:
            x = (x * f).clamp(0, 1) if k == 'b' else ((x - x.mean()) * f + x.mean()).clamp(0, 1)
        out.append(x)
    return torch.stack(out)

def to_input(idx, train, rng=None):
    x = X[idx].float() / 255.0                                       # B x 2 x H x W
    a, p = x[:, :1], x[:, 1:]
    if train:
        a, p = augment(a, rng), augment(p, rng)
    norm = lambda t: (t.repeat(1, 3, 1, 1) - MEAN) / STD
    return norm(a), norm(p)

class Pooled(nn.Module):
    """Wraps any feature extractor (e.g. RadImageNet full-model object) so that it returns B x C."""
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, x):
        y = self.m(x)
        if isinstance(y, (tuple, list)):
            y = y[0]
        if hasattr(y, 'logits'):
            y = y.logits
        return F.adaptive_avg_pool2d(y, 1).flatten(1) if y.dim() == 4 else y.flatten(1)

_RAD_CACHE = {}
def radimagenet_backbone():
    if 'm' not in _RAD_CACHE:
        if LOCAL_TEST:
            path = BASE + '/radimagenet_test.pt'
        else:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download(repo_id=RAD_REPO, filename=RAD_FILE)
        obj = torch.load(path, map_location='cpu', weights_only=False)
        if isinstance(obj, dict):                                       # file is a state_dict -> rebuild Inception-v3 and load it
            sd = obj.get('state_dict', obj.get('model', obj))
            for pre in ['module.', 'backbone.', 'model.', 'base_model.', 'features.']:
                if all(k.startswith(pre) for k in sd):
                    sd = {k[len(pre):]: v for k, v in sd.items()}
            print('RadImageNet file is a state_dict with', len(sd), 'tensors, e.g.', list(sd)[:3])
            best = None
            for builder in ['timm', 'torchvision']:
                cand = timm.create_model('inception_v3', pretrained=False, num_classes=0) if builder == 'timm' else __import__('torchvision').models.inception_v3(weights=None, aux_logits=False, init_weights=False)
                msd = cand.state_dict()
                ok = {k: v for k, v in sd.items() if k in msd and msd[k].shape == v.shape}
                isbb = lambda k: not k.startswith(('fc.', 'AuxLogits'))
                frac = sum(v.numel() for k, v in ok.items() if isbb(k)) / sum(v.numel() for k, v in msd.items() if isbb(k))
                print(f'  {builder} inception_v3: {len(ok)}/{len(msd)} tensors match ({frac:.0%} of backbone weights)')
                if best is None or frac > best[0]:
                    best = (frac, builder, cand, ok)
            frac, builder, m, ok = best
            if frac < 0.95:                                             # saved as nn.Sequential ('0.conv.weight', ...): map by position + shape
                cand = timm.create_model('inception_v3', pretrained=False, num_classes=0); msd = cand.state_dict()
                tk = [k for k in msd if not k.startswith(('fc.', 'AuxLogits'))]
                sv = list(sd.values())
                if len(tk) == len(sv) and all(msd[a].shape == v.shape for a, v in zip(tk, sv)):
                    m, ok, frac, builder = cand, dict(zip(tk, sv)), 1.0, 'timm (by position)'
                    print(f'  positional mapping: {len(tk)}/{len(tk)} tensors, all shapes match')
            assert frac > 0.95, ('RadImageNet weights do not match Inception-v3 (only %.0f%% loaded) - check the RadImageNet weight file' % (100 * frac))
            m.load_state_dict(ok, strict=False)
            print(f'  loaded into {builder} inception_v3 ({frac:.0%} of weights)')
        elif isinstance(obj, torch.jit.ScriptModule):
            m = obj
        else:
            m = obj
        for attr in ['fc', 'classifier', 'head']:                       # drop a classification head if present
            if hasattr(m, attr) and isinstance(getattr(m, attr), nn.Module):
                setattr(m, attr, nn.Identity())
        if hasattr(m, 'aux_logits'):
            m.aux_logits = False; m.AuxLogits = None
        _RAD_CACHE['m'] = m
        print('RadImageNet backbone:', type(m).__name__, '| file:', path)
    return Pooled(copy.deepcopy(_RAD_CACHE['m']))

def make_backbone(name, init):
    if init == 'RAD':
        return radimagenet_backbone()
    bb = timm.create_model(name, pretrained=PRETRAINED and init != 'SCRATCH', num_classes=0)
    if init == 'IN':
        return bb
    f = (f'{BSD}/model_bs80k_{name}.pt' if init == 'BS' else
         f'{OUT}/models/model_bs80k_{name}_frac{init[3:]}.pt' if init.startswith('BSf') else       # 'BSf0.10' (independent subsets)
         f'{OUT}/models/model_bs80k_{name}_nested{init[3:]}.pt')                                     # 'BSn0.10' (nested subsets, equal steps)
    sd = torch.load(f, map_location='cpu')
    res = bb.load_state_dict({k[len('backbone.'):]: v for k, v in sd.items() if k.startswith('backbone.')}, strict=False)
    assert not res.missing_keys, f'missing keys in {f}: {res.missing_keys[:3]}'
    return bb

class DualViewNet(nn.Module):
    def __init__(self, name, n, init):
        super().__init__()
        self.backbone = make_backbone(name, init)
        self.backbone.eval()
        with torch.no_grad():
            c = self.backbone(torch.zeros(1, 3, H, W)).shape[1]
        self.head = nn.Sequential(nn.Dropout(0.4), nn.Linear(2 * c, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n))
    def features(self, a, p):
        return torch.cat([self.backbone(a), self.backbone(p)], 1)
    def forward(self, a, p):
        return self.head(self.features(a, p))

def predict(model, idx, amp, feats=False):
    model.eval(); P, Fz = [], []
    with torch.no_grad(), torch.autocast(device_type='cuda', enabled=amp):
        for i in range(0, len(idx), 32):
            a, p = to_input(idx[i:i + 32], False)
            z = model.features(a, p); P.append(F.softmax(model.head(z).float(), 1).cpu())
            if feats:
                Fz.append(z.float().cpu())
    P = torch.cat(P).numpy() if P else np.zeros((0, 1))
    return (P, torch.cat(Fz).numpy().astype(np.float32)) if feats else P

def macro_f1(pred, y, n):
    f = []
    for c in range(n):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))

def train(name, init, n, tr_idx, tr_y, va_idx, va_y, seed, epochs=EPOCHS, patience=PATIENCE, amp=True):
    set_seed(seed); amp = amp and DEVICE == 'cuda'
    model = DualViewNet(name, n, init).to(DEVICE)
    cnt = np.bincount(tr_y, minlength=n)
    w = torch.tensor(len(tr_y) / (n * np.maximum(cnt, 1)), dtype=torch.float32, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    opt = torch.optim.AdamW([{'params': model.backbone.parameters(), 'lr': 1e-4}, {'params': model.head.parameters(), 'lr': 1e-3}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=amp)
    rng = np.random.default_rng(seed); yt = torch.tensor(tr_y, device=DEVICE)
    best, state, wait, best_ep, ep = -1.0, None, 0, 0, 0
    for ep in range(1, epochs + 1):
        model.train()
        perm = rng.permutation(len(tr_idx)); nb = max(len(perm) // BATCH, 1)
        for b in range(nb):
            j = perm[b * BATCH:(b + 1) * BATCH]
            a, p = to_input(tr_idx[j], True, rng)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', enabled=amp):
                loss = loss_fn(model(a, p), yt[j])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        pv = predict(model, va_idx, amp)
        vf = macro_f1(pv.argmax(1), va_y, n) if not np.isnan(pv).any() else -1
        if vf > best:
            best, state, wait, best_ep = vf, copy.deepcopy(model.state_dict()), 0, ep
        else:
            wait += 1
            if wait >= patience:
                break
    if state is not None:
        model.load_state_dict(state)
    return model, dict(best_inner_f1=best, best_epoch=best_ep, epochs_run=ep, amp=amp)

def ids_to_idx(ids):
    return np.array([IDX[p] for p in ids])

def save_preds(path, P, ids, y=None):
    d = pd.DataFrame(P, columns=[f'prob_{i}' for i in range(P.shape[1])])
    if y is not None:
        d.insert(0, 'y_true', y)
    d.insert(0, 'patient_id', list(ids)); d.insert(1, 'camera', [CAM[p] for p in ids]); d.to_csv(path, index=False)

def done_keys():
    return {f[:-5] for f in os.listdir(f'{OUT}/logs') if f.endswith('.json')}

def log_run(row):
    # one small JSON per finished run (safe if Colab disconnects); runs.csv is rebuilt from all of them
    with open(f"{OUT}/logs/{row['key']}.json", 'w') as fh:
        json.dump({k: (v.item() if hasattr(v, 'item') else v) for k, v in row.items()}, fh)
    rows = [json.load(open(f'{OUT}/logs/{f}')) for f in sorted(os.listdir(f'{OUT}/logs')) if f.endswith('.json')]
    pd.DataFrame(rows).to_csv(RUNS, index=False)

def time_left():
    return MAX_HOURS - (time.time() - T_START) / 3600

def run_job(key, exp, name, init, n, seed, tr, va, evals, feat_sets=(), extra=None):
    """tr/va: DataFrames with patient_id and y. evals: {kind: (ids, y or None)}. Saves preds of every eval set."""
    t0 = time.time()
    for amp in [True, False]:
        model, info = train(name, init, n, ids_to_idx(tr.patient_id), tr.y.values, ids_to_idx(va.patient_id), va.y.values, seed, amp=amp)
        outs, bad = {}, False
        for kind, (ids, y) in evals.items():
            res = predict(model, ids_to_idx(ids), info['amp'], feats=kind in feat_sets)
            P = res[0] if kind in feat_sets else res
            bad |= bool(np.isnan(P).any()); outs[kind] = res
        if not bad or not amp or DEVICE != 'cuda':
            break
        print('  NaN -> retrying in full precision'); del model; gc.collect(); torch.cuda.empty_cache()
    for kind, (ids, y) in evals.items():
        res = outs[kind]; P = res[0] if kind in feat_sets else res
        save_preds(f'{OUT}/preds/{key}_{kind}.csv', P, ids, y)
        if kind in feat_sets:
            np.savez_compressed(f'{OUT}/feats/{key}_{kind}.npz', ids=np.array(ids), z=np.clip(res[1], -6e4, 6e4).astype(np.float16))
    row = dict(key=key, exp=exp, model=name, init=init, seed=seed, minutes=round((time.time() - t0) / 60, 2), n_train=len(tr), n_inner=len(va), **info, **(extra or {}))
    for kind, (ids, y) in evals.items():
        if y is not None:
            res = outs[kind]; P = res[0] if kind in feat_sets else res
            row[f'{kind}_f1'] = macro_f1(P.argmax(1), np.asarray(y), n)
    log_run(row)
    print(f"  {key}: inner {info['best_inner_f1']:.3f} | " + ' | '.join(f"{k[:-3]} {v:.3f}" for k, v in row.items() if k.endswith('_f1') and k != 'best_inner_f1') + f" | {row['minutes']} min", flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()

def inner_split(df, seed=0):
    for strat in [df.label + '_' + df.machine, df.label, None]:     # class x camera; fall back if a stratum is too small
        try:
            tr, va = train_test_split(df, test_size=INNER_FRAC, stratify=strat, random_state=seed)
            return tr.reset_index(drop=True), va.reset_index(drop=True)
        except ValueError:
            continue

def eta(exp, remaining):
    if not os.path.exists(RUNS):
        return ''
    r = pd.read_csv(RUNS); r = r[r.exp == exp] if 'exp' in r else r.iloc[:0]
    return f'(~{remaining * r.minutes.mean() / 60:.1f} h left for {exp})' if len(r) else ''
print('Ready')

In [ ]:
# Experiments 1 + 2: train inside one camera, test on the held-out fold (within) and on the other camera (cross)
def inner_seed(k, seed):
    return 1000 + k + 100 * (seed - 42)          # identical in all notebooks

PATIENTS = sp.copy()                              # final dataset (notebook 02); every patient has a within-machine fold
for c in CAMERAS:
    pc = PATIENTS[(PATIENTS.machine == c) & (PATIENTS.split != 'test')]
    print(f'{c}: pool {len(pc)} (per fold {pc.fold.value_counts().sort_index().tolist()}), '
          f'test {((PATIENTS.machine == c) & (PATIENTS.split == "test")).sum()}, classes {pc.label.value_counts().to_dict()}')

def machine_jobs():
    jobs = []
    for seed in SEEDS:                           # seed 42 for everything first
        for init in INITS:
            for task in TASKS:
                for name in MODELS:
                    for cam in CAMERAS:
                        for k in QUICK_FOLDS:
                            jobs.append((f'mach_{task}_{name}_{init}_{cam}_s{seed}_fold{k}', task, name, init, cam, seed, k))
    return jobs

def run_machine():
    jobs = machine_jobs(); done = done_keys(); todo = [j for j in jobs if j[0] not in done]
    print(f'machine: {len(jobs) - len(todo)}/{len(jobs)} done', eta('machine', len(todo)))
    for key, task, name, init, cam, seed, k in todo:
        if time_left() <= 0:
            print('MAX_HOURS reached - restart the runtime and run all again to resume'); return False
        mp = TASKS[task]; n = len(set(mp.values()))
        d = PATIENTS[PATIENTS.label.isin(mp)].assign(y=lambda x: x.label.map(mp))
        src_pool = d[(d.machine == cam) & (d.split != 'test')]
        tr_all, outer = src_pool[src_pool.fold != k], src_pool[src_pool.fold == k]
        tr, inner = inner_split(tr_all, seed=inner_seed(k, seed))
        src_test = d[(d.machine == cam) & (d.split == 'test')]
        tgt = d[d.machine != cam]
        assert not set(tr.patient_id) & (set(outer.patient_id) | set(tgt.patient_id) | set(src_test.patient_id) | set(inner.patient_id))
        assert set(tr.patient_id) | set(inner.patient_id) <= set(src_pool.patient_id) and (tr.machine == cam).all()
        evals = {'oof': (outer.patient_id, outer.y.values), 'inner': (inner.patient_id, inner.y.values), 'cross': (tgt.patient_id, tgt.y.values)}
        if len(src_test):
            evals['srctest'] = (src_test.patient_id, src_test.y.values)
        print(f'[machine] {task} | {name} | {init} | train {cam} | seed {seed} | fold {k}  '
              f'(train {len(tr)}, inner {len(inner)}, within {len(outer)}, cross {len(tgt)})')
        run_job(key, 'machine', name, init, n, seed, tr, inner, evals,
                extra=dict(task=task, camera=cam, fold=k, inner_seed=inner_seed(k, seed)))
    return True

OK = run_machine() if RUN_TRAINING else True
if os.path.exists(RUNS):
    r = pd.read_csv(RUNS)
    print(r.groupby('exp').agg(runs=('key', 'count'), minutes=('minutes', 'sum')).assign(hours=lambda d: (d.minutes / 60).round(1)))
print('ALL RUNS FINISHED' if OK else 'NOT FINISHED YET - Run all again (finished runs are skipped); the analysis below uses what is done')

In [ ]:
# Analysis 1: collect predictions (within = seed-averaged OOF; cross = each target patient scored by one fold model per seed)
def rd(path):
    return pd.read_csv(path) if os.path.exists(path) else None

def pcols(df):
    return sorted([c for c in df.columns if c.startswith('prob_')], key=lambda s: int(s[5:]))

# every target patient of the other camera is assigned to ONE fold model (pool patients: their own frozen fold;
# test patients: sorted id order modulo 5), so within and cross predictions are averages over the same number of models (one per seed)
ASSIGN = {}
for c in CAMERAS:
    d = PATIENTS[PATIENTS.machine == c]
    for p, f in zip(d.patient_id, d.fold):
        if not np.isnan(f):
            ASSIGN[p] = int(f)
    for i, p in enumerate(sorted(d[d.split == 'test'].patient_id)):
        ASSIGN[p] = i % N_FOLDS

def collect(task, name, init, cam):
    """returns dict with within / cross / cross_ens15 / srctest DataFrames (patient_id, y_true, prob_*) or None if incomplete"""
    seeds_ok = []
    for s in SEEDS:
        if all(os.path.exists(f'{OUT}/preds/mach_{task}_{name}_{init}_{cam}_s{s}_fold{k}_cross.csv') for k in QUICK_FOLDS):
            seeds_ok.append(s)
    if not seeds_ok:
        return None
    W, X_, E, S = [], [], [], []
    for s in seeds_ok:
        oof = pd.concat([rd(f'{OUT}/preds/mach_{task}_{name}_{init}_{cam}_s{s}_fold{k}_oof.csv') for k in QUICK_FOLDS])
        W.append(oof.sort_values('patient_id').reset_index(drop=True))
        cr = {k: rd(f'{OUT}/preds/mach_{task}_{name}_{init}_{cam}_s{s}_fold{k}_cross.csv').sort_values('patient_id').reset_index(drop=True) for k in QUICK_FOLDS}
        base = cr[QUICK_FOLDS[0]][['patient_id', 'camera', 'y_true']].copy(); pc = pcols(cr[QUICK_FOLDS[0]])
        assigned = np.stack([cr[ASSIGN[p]].loc[i, pc].values for i, p in enumerate(base.patient_id)])
        X_.append(base.assign(**{c: assigned[:, j] for j, c in enumerate(pc)}))
        E += [cr[k] for k in QUICK_FOLDS]
        S += [rd(f).sort_values('patient_id').reset_index(drop=True) for f in [f'{OUT}/preds/mach_{task}_{name}_{init}_{cam}_s{s}_fold{k}_srctest.csv' for k in QUICK_FOLDS] if os.path.exists(f)]
    def avg(lst):
        out = lst[0][['patient_id', 'camera', 'y_true']].copy(); pc = pcols(lst[0])
        for x in lst:
            assert (x.patient_id.values == out.patient_id.values).all()
        out[pc] = np.mean([x[pc].values for x in lst], 0); return out
    res = dict(within=avg(W), cross=avg(X_), cross_ens15=avg(E), srctest=avg(S) if S else None, seeds=seeds_ok)
    assert (res['within'].camera == cam).all() and (res['cross'].camera != cam).all()
    return res

def combined_ref(task, name, init, cam):
    """the earlier revision run combined-machine model (trained on both cameras): seed-averaged OOF on camera `cam` pool patients"""
    L = []
    if not TIER2:
        return None
    for s in SEEDS:
        fs = [f'{TIER2}/preds/nested_{task}_{name}_{init}_s{s}_fold{k}_oof.csv' for k in range(N_FOLDS)]
        if all(os.path.exists(f) for f in fs):
            L.append(pd.concat([pd.read_csv(f) for f in fs]).sort_values('patient_id').reset_index(drop=True))
    if not L:
        return None
    out = L[0][['patient_id', 'camera', 'y_true']].copy(); pc = pcols(L[0]); out[pc] = np.mean([x[pc].values for x in L], 0)
    return out[out.camera == cam].reset_index(drop=True)

GROUPS = [(t, m, i, c) for t in TASKS for m in MODELS for i in INITS for c in CAMERAS]
DATA = {}
for g in GROUPS:
    r_ = collect(*g)
    if r_ is not None:
        DATA[g] = r_
print(f'complete groups: {len(DATA)}/{len(GROUPS)}')

In [ ]:
# Analysis 2: metrics + joint patient bootstrap
from sklearn.metrics import roc_auc_score
cam_patients = {c: sorted(PATIENTS[PATIENTS.machine == c].patient_id) for c in CAMERAS}
cam_pos = {c: {p: i for i, p in enumerate(cam_patients[c])} for c in CAMERAS}
rng = np.random.default_rng(2026)
BOOT = {c: np.stack([np.bincount(rng.integers(0, len(cam_patients[c]), len(cam_patients[c])), minlength=len(cam_patients[c]))
                     for _ in range(N_BOOT)]).astype(np.float32) for c in CAMERAS}      # one resample of each camera's patients, shared by all

def weighted_scores(df, cam, wts):
    """macro-F1 and balanced accuracy for each row of weights (argmax predictions)"""
    pc = pcols(df); y = df.y_true.values.astype(int); pr = df[pc].values.argmax(1); n = len(pc)
    Wm = wts[:, [cam_pos[cam][p] for p in df.patient_id]]
    f1, rec = [], []
    for j in range(n):
        tp = Wm @ ((pr == j) & (y == j)).astype(np.float32); fp = Wm @ ((pr == j) & (y != j)).astype(np.float32)
        fn = Wm @ ((pr != j) & (y == j)).astype(np.float32)
        f1.append(np.where(tp > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1e-9), 0.0)); rec.append(tp / np.maximum(tp + fn, 1e-9))
    return np.mean(f1, 0), np.mean(rec, 0)

def point(df):
    pc = pcols(df); y = df.y_true.values.astype(int); P = df[pc].values; pr = P.argmax(1); n = len(pc)
    f1 = macro_f1(pr, y, n); rec = [np.mean(pr[y == j] == j) if (y == j).any() else np.nan for j in range(n)]
    try:
        auc = roc_auc_score(y, P[:, 1]) if n == 2 else roc_auc_score(y, P, multi_class='ovr', average='macro')
    except ValueError:
        auc = np.nan
    cm = pd.crosstab(pd.Series(y, name='true'), pd.Series(pr, name='pred')).reindex(index=range(n), columns=range(n), fill_value=0)
    out = dict(n=len(y), f1=f1, bacc=float(np.nanmean(rec)), auc=auc, cm=cm.values.tolist())
    if n == 2:
        out.update(sens=rec[1], spec=rec[0])
    else:
        out.update({f'recall_{j}': rec[j] for j in range(n)})
    return out

def pval(d):
    return min(1.0, 2 * min((np.sum(d <= 0) + 1) / (len(d) + 1), (np.sum(d >= 0) + 1) / (len(d) + 1)))

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for r_, i in enumerate(o):
        run = max(run, (m - r_) * p[i]); adj[i] = min(1.0, run)
    return adj

ROWS, DIST, CMS = [], {}, []
for (task, name, init, cam), r_ in DATA.items():
    tgt = 'M2' if cam == 'M1' else 'M1'
    w, x = r_['within'], r_['cross']
    pw, px, pe = point(w), point(x), point(r_['cross_ens15']); ps = point(r_['srctest']) if r_['srctest'] is not None else dict(f1=np.nan, n=0)
    bw, _ = weighted_scores(w, cam, BOOT[cam]); bx, _ = weighted_scores(x, tgt, BOOT[tgt])
    drop = bx - bw; DIST[(task, name, init, cam)] = (bw, bx)
    comb = combined_ref(task, name, init, tgt)
    row = dict(task=TASK_ID[task], task_key=task, model=MN[name], init=init, train_camera=cam, test_camera_cross=tgt, seeds=len(r_['seeds']),
               n_within=pw['n'], n_cross=px['n'],
               within_f1=pw['f1'], within_lo=np.percentile(bw, 2.5), within_hi=np.percentile(bw, 97.5),
               cross_f1=px['f1'], cross_lo=np.percentile(bx, 2.5), cross_hi=np.percentile(bx, 97.5),
               drop=px['f1'] - pw['f1'], drop_lo=np.percentile(drop, 2.5), drop_hi=np.percentile(drop, 97.5), p=pval(drop),
               within_bacc=pw['bacc'], cross_bacc=px['bacc'], within_auc=pw['auc'], cross_auc=px['auc'],
               cross_ens15_f1=pe['f1'], srctest_f1=ps['f1'], n_srctest=ps['n'],
               combined_f1_on_cross_camera=(point(comb)['f1'] if comb is not None and len(comb) else np.nan))
    for k_ in ['sens', 'spec', 'recall_0', 'recall_1', 'recall_2']:
        if k_ in pw:
            row[f'within_{k_}'] = pw[k_]; row[f'cross_{k_}'] = px[k_]
    ROWS.append(row)
    for setname, pt in [('within', pw), ('cross', px)]:
        CMS.append(dict(task=TASK_ID[task], model=MN[name], init=init, train_camera=cam, set=setname, confusion_matrix=json.dumps(pt['cm'])))
A = pd.DataFrame(ROWS)
if len(A):
    A['p_holm'] = np.nan
    for ini in A.init.unique():                   # family: all task x backbone x direction drops of one initialisation
        k_ = A.init == ini; A.loc[k_, 'p_holm'] = holm(A.loc[k_, 'p'].values)
    A = A.sort_values(['init', 'task', 'model', 'train_camera']).reset_index(drop=True)
    A.to_csv(f'{OUT}/analysis/A_within_vs_cross.csv', index=False); pd.DataFrame(CMS).to_csv(f'{OUT}/analysis/A_confusion_matrices.csv', index=False)
    show = ['task', 'model', 'init', 'train_camera', 'seeds', 'within_f1', 'cross_f1', 'drop', 'drop_lo', 'drop_hi', 'p_holm', 'within_auc', 'cross_auc', 'combined_f1_on_cross_camera']
    print(A[show].round(3).to_string(index=False))

In [ ]:
# Analysis 3: pooled drop per initialisation and direction, and ImageNet vs BS-80K (joint bootstrap, same resamples)
P_ROWS = []
def pooled(keys):
    bw = np.mean([DIST[k][0] for k in keys], 0); bx = np.mean([DIST[k][1] for k in keys], 0)
    pw_ = np.mean([A[(A.task_key == k[0]) & (A.model == MN[k[1]]) & (A.init == k[2]) & (A.train_camera == k[3])].within_f1.iloc[0] for k in keys])
    px_ = np.mean([A[(A.task_key == k[0]) & (A.model == MN[k[1]]) & (A.init == k[2]) & (A.train_camera == k[3])].cross_f1.iloc[0] for k in keys])
    return pw_, px_, bw, bx

for ini in INITS:
    for direction in ['M1', 'M2', 'both']:
        keys = [k for k in DIST if k[2] == ini and (direction == 'both' or k[3] == direction)]
        if not keys:
            continue
        pw_, px_, bw, bx = pooled(keys); d = bx - bw
        sub = A[(A.init == ini) & ((A.train_camera == direction) if direction != 'both' else A.train_camera.notna())]
        P_ROWS.append(dict(contrast=f'cross − within ({ini})', direction={'M1': 'M1 → M2', 'M2': 'M2 → M1', 'both': 'both directions'}[direction],
                           n_pairs=len(keys), within_f1=pw_, cross_f1=px_, mean_drop=px_ - pw_, lo=np.percentile(d, 2.5), hi=np.percentile(d, 97.5),
                           p=pval(d), n_drop_negative=int((sub['drop'] < 0).sum()), n_sig_holm=int((sub.p_holm < 0.05).sum())))
for direction in ['M1', 'M2', 'both']:
    kb = [k for k in DIST if k[2] == 'BS' and (direction == 'both' or k[3] == direction)]
    ki = [(k[0], k[1], 'IN', k[3]) for k in kb]
    if not kb or not all(k in DIST for k in ki):
        continue
    _, _, bwb, bxb = pooled(kb); _, _, bwi, bxi = pooled(ki); dd = (bxb - bwb) - (bxi - bwi)
    est = np.mean([A[(A.task_key == k[0]) & (A.model == MN[k[1]]) & (A.init == 'BS') & (A.train_camera == k[3])]['drop'].iloc[0] -
                   A[(A.task_key == k[0]) & (A.model == MN[k[1]]) & (A.init == 'IN') & (A.train_camera == k[3])]['drop'].iloc[0] for k in kb])
    P_ROWS.append(dict(contrast='drop(BS-80K) − drop(ImageNet)', direction={'M1': 'M1 → M2', 'M2': 'M2 → M1', 'both': 'both directions'}[direction],
                       n_pairs=len(kb), mean_drop=est, lo=np.percentile(dd, 2.5), hi=np.percentile(dd, 97.5), p=pval(dd)))
PO = pd.DataFrame(P_ROWS)
if len(PO):
    PO.to_csv(f'{OUT}/analysis/B_pooled.csv', index=False); print(PO.round(3).to_string(index=False))
    print('\nPositive "drop" = better on the other camera; negative = worse. Joint bootstrap over patients of each camera,', N_BOOT, 'resamples.')

In [ ]:
# Analysis 4: Word report + short text summary
from docx import Document
from docx.shared import Pt, Inches
def tbl(doc, df, cols, heads, fmt=3, size=7.5):
    t = doc.add_table(rows=1, cols=len(cols)); t.style = 'Table Grid'
    for i, h in enumerate(heads):
        c = t.rows[0].cells[i]; c.text = ''; r = c.paragraphs[0].add_run(h); r.bold = True; r.font.size = Pt(size)
    for _, row in df.iterrows():
        cs = t.add_row().cells
        for i, c in enumerate(cols):
            v = row[c] if not callable(c) else c(row)
            s = (f'{v:.{fmt}f}' if isinstance(v, (float, np.floating)) and not np.isnan(v) else ('–' if isinstance(v, (float, np.floating)) else str(v)))
            cs[i].text = ''; cs[i].paragraphs[0].add_run(s).font.size = Pt(size)
    doc.add_paragraph()
if len(A):
    doc = Document(); doc.styles['Normal'].font.name = 'Calibri'; doc.styles['Normal'].font.size = Pt(10)
    doc.add_heading('Machine shift: within-machine vs cross-machine performance (notebook 03)', 0)
    doc.add_paragraph(f'Runs complete for {len(DATA)}/{len(GROUPS)} task × backbone × initialisation × camera groups. '
                      'Within = 5-fold CV inside the training camera (seed-averaged out-of-fold predictions). Cross = the same fold models applied '
                      'to every patient of the other camera; each target patient is scored by one fold model per seed so that within and cross '
                      'predictions average the same number of models. Drop = cross − within macro-F1; 95 % CI and two-sided p from a joint patient '
                      f'bootstrap ({N_BOOT} resamples of each camera, shared across all models); Holm correction within each initialisation. '
                      'Combined = the earlier revision run model trained on both cameras (about twice the training data), evaluated on the same cross-camera pool patients.')
    ci = lambda a, b, c: (lambda r: f"{r[a]:.3f} [{r[b]:.3f}, {r[c]:.3f}]")
    doc.add_heading('1. Pooled drop', 1)
    if len(PO):
        tbl(doc, PO, ['contrast', 'direction', 'n_pairs', 'within_f1', 'cross_f1', lambda r: f"{r['mean_drop']:+.3f} [{r['lo']:+.3f}, {r['hi']:+.3f}]", 'p'],
            ['Contrast', 'Direction', 'Pairs', 'Within F1', 'Cross F1', 'Mean Δ [95 % CI]', 'p'])
    doc.add_heading('2. Every task × backbone × direction', 1)
    tbl(doc, A, ['task', 'model', 'init', lambda r: f"{r['train_camera']}→{r['test_camera_cross']}", 'seeds', ci('within_f1', 'within_lo', 'within_hi'),
                 ci('cross_f1', 'cross_lo', 'cross_hi'), lambda r: f"{r['drop']:+.3f} [{r['drop_lo']:+.3f}, {r['drop_hi']:+.3f}]", 'p_holm', 'combined_f1_on_cross_camera'],
        ['Task', 'Backbone', 'Init', 'Train→Test', 'Seeds', 'Within F1 [CI]', 'Cross F1 [CI]', 'Drop [CI]', 'p Holm', 'Combined F1'], size=7)
    doc.add_heading('3. Balanced accuracy, AUC, sensitivity and specificity', 1)
    bcols = ['task', 'model', 'init', 'train_camera', 'within_bacc', 'cross_bacc', 'within_auc', 'cross_auc'] + [c for c in ['within_sens', 'cross_sens', 'within_spec', 'cross_spec'] if c in A]
    tbl(doc, A, bcols, ['Task', 'Backbone', 'Init', 'Train', 'Within BAcc', 'Cross BAcc', 'Within AUC', 'Cross AUC'] + [c.replace('_', ' ') for c in bcols[8:]], size=7)
    t1 = A[A.task == 'T1']
    if len(t1):
        doc.add_heading('4. Three-class task: per-class recall (0 = Met, 1 = Deg, 2 = Nor)', 1)
        tbl(doc, t1, ['model', 'init', 'train_camera'] + [f'{s}_recall_{j}' for j in range(3) for s in ['within', 'cross']],
            ['Backbone', 'Init', 'Train'] + [f'{s} recall {j}' for j in range(3) for s in ['within', 'cross']], size=7)
    doc.add_heading('5. Extra checks', 1)
    tbl(doc, A, ['task', 'model', 'init', 'train_camera', 'cross_f1', 'cross_ens15_f1', 'n_srctest', 'srctest_f1'],
        ['Task', 'Backbone', 'Init', 'Train', 'Cross F1 (1 model/seed)', 'Cross F1 (15-model ensemble)', 'n own test', 'Own locked test F1'], size=7)
    doc.add_paragraph('Confusion matrices for every row: analysis/A_confusion_matrices.csv.')
    doc.save(f'{OUT}/analysis/Machine_Shift_Report.docx')
    lines = [f'notebook 03 machine shift - {len(DATA)}/{len(GROUPS)} groups complete, bootstrap {N_BOOT}']
    for _, r in PO.iterrows():
        lines.append(f"{r['contrast']} | {r['direction']} | Δ {r['mean_drop']:+.3f} [{r['lo']:+.3f}, {r['hi']:+.3f}] p {r['p']:.4f}")
    open(f'{OUT}/analysis/summary.txt', 'w').write('\n'.join(lines)); print('\n'.join(lines))
    print('Saved:', f'{OUT}/analysis/Machine_Shift_Report.docx')